# Agent高级用法-流式输出
## 1、values的输出模型

In [ ]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint
@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息
    Args:
    customer_id: 客户ID，用于唯一标识客户
    Returns:
    包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}
@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史
    Args:
    customer_id: 客户ID，用于唯一标识客户
    Returns:
    包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}
@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动
    Returns:
    包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
    "promotions": ["老用户优惠", "会员专属折扣"],
    "valid_until": "2027-01-31"
    }
    # 创建客户服务Agent
customer_service_agent = create_agent(
model=model,
tools=[query_customer_data, check_order_history, get_current_promotions]
)
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="values"
    ):
    rprint(chunk)
print("-" * 50)

## 2、update的输出模式（默认输出格式）

In [ ]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="updates"
    ):
    rprint(chunk)
print("-" * 50)

## 3、messages输出模式

In [ ]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="messages"
    ):
    # rprint(chunk)
print(chunk[0].content,end="",flush=True)
print("-" * 50)

## 4、tasks输出模式

In [ ]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="tasks"
    ):
    rprint(chunk)
print("-" * 50)

## 5、debug输出模式

In [ ]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="debug"
    ):
    rprint(chunk)
print("-" * 50)